# Prepare training data and held-out evaluation
    Default: paper-stated settings plus the shared assumptions in docs/ASSUMPTIONS.md.
    The short pilot uses the same algorithm
    and memory requirements, with fewer updates. Generate one held-out GPT-6.1 Sol
    corpus (the user-selected change from the paper's GPT-5), or import existing
    held-out examples. Both conditions use the same corpus.
    See `docs/CORPUS.md` for the paper's procedure and explicit construction choices.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile

candidates = [Path(os.environ.get("GASP_ROOT", ".")), Path.cwd(), Path.cwd().parent, Path("/content/gasp")]
ROOT = next((p.resolve() for p in candidates if (p / "src/gasp").is_dir()), None)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Unzip dist/gasp-colab.zip and open this notebook inside the project.")
    print("Select the delivered gasp-colab.zip archive.")
    uploaded = files.upload()
    archives = [p for p in uploaded if p.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Select exactly one project archive")
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            if not name.startswith("gasp/") or ".." in Path(name).parts:
                raise ValueError("Unexpected archive member")
        z.extractall("/content")
    ROOT = Path("/content/gasp")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

def command(*args):
    result = subprocess.run([sys.executable, *map(str,args)], cwd=ROOT,
                            env={**os.environ,"PYTHONPATH":str(ROOT/"src")},
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()

print("Project:", ROOT)


In [ ]:
# Run once per new Colab runtime. Restart the runtime afterward, then rerun bootstrap.
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    assert (3,11) <= sys.version_info[:2] < (3,14), "Use Python 3.11–3.13"
    # These optional image/audio packages can conflict with a pinned text-only torch.
    command("-m", "pip", "uninstall", "-y", "torchvision", "torchaudio")
    command("-m", "pip", "install", "-r", "requirements-colab.txt")
    command("-m", "pip", "install", "--no-deps", ".")
    print("Restart runtime if packages were already imported. No training has started.")


In [ ]:
# Optional persistence. Keep downloaded HF weights in Colab scratch; persist data/runs.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/gasp-experiment")
else:
    WORK = ROOT
WORK.mkdir(parents=True, exist_ok=True)
print("Data and outputs:", WORK)


In [ ]:
import torch
from gasp.config import Config
PROFILE = "full"  # "pilot" only shortens training; it does not enable LoRA
assert PROFILE in ("pilot", "full")
cfg = Config.load(ROOT/f"configs/{PROFILE}.json")
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Use a bf16-capable GPU; notebook00 is CPU-only"
print(torch.cuda.get_device_name(), torch.cuda.get_device_properties(0).total_memory/2**30, "GiB")
RAW = WORK/"data/raw"
DATA = WORK/f"data/{PROFILE}-v4"
print("Immutable cache version:", DATA)


In [ ]:
if not (RAW/"math_raw.jsonl").exists():
    command("-m", "gasp", "download", "--output", RAW)
if not (DATA/"train.jsonl").exists():
    command("-m", "gasp", "prepare-train", "--config", ROOT/f"configs/{PROFILE}.json",
            "--source", RAW/"math_raw.jsonl", "--output", DATA/"train.jsonl",
            "--candidates", 7500)
print(json.loads(Path(str(DATA/"train.jsonl")+".manifest.json").read_text()))


In [ ]:
import shlex
# Enable after setting OPENAI_API_KEY in this runtime. The API stage is paid.
RUN_CORPUS_GENERATION = False
MAX_NEW_REQUESTS = 4  # Inspect a first batch, then set None and rerun to finish.
CONTEXTS = DATA/"eval-contexts.jsonl"
EXTERNAL = DATA/"held-out-corruptions.jsonl"
PROVENANCE = Path(str(EXTERNAL)+".provenance.json")
prepare = ["-m", "gasp", "prepare-eval-contexts", "--config", DATA/"resolved_config.json",
           "--source", RAW/"gsm_raw.jsonl", "--output", CONTEXTS]
generate = ["-m", "gasp", "generate-eval", "--source", CONTEXTS, "--output", EXTERNAL]
if MAX_NEW_REQUESTS is not None:
    generate += ["--max-requests", MAX_NEW_REQUESTS]
for args in (prepare, generate):
    print(shlex.join([sys.executable, *map(str,args)]))
if RUN_CORPUS_GENERATION and not EXTERNAL.exists():
    if not CONTEXTS.exists():
        command(*prepare)
    command(*generate)


In [ ]:
# Inspect generated clean/polluted pairs first. For an existing author corpus,
# set EXTERNAL and PROVENANCE above to its paths instead.
if not (DATA/"eval.jsonl").exists():
    if not EXTERNAL.exists() or not PROVENANCE.exists():
        raise FileNotFoundError("Finish corpus generation or supply a held-out corpus; see docs/CORPUS.md")
    command("-m", "gasp", "import-eval", "--config", DATA/"resolved_config.json",
            "--source", EXTERNAL, "--provenance", PROVENANCE,
            "--output", DATA/"eval.jsonl")
from gasp.data import read_cache, assert_disjoint
train, tm = read_cache(DATA/"train.jsonl", "train")
evaluation, em = read_cache(DATA/"eval.jsonl", "eval")
assert_disjoint(train, evaluation)
print("Training questions:", len(train), "Evaluation contexts:", len(evaluation))
print("Protocol:", em["provenance"])
print("Fixed clean-solved contexts:", sum(r["initial_clean_solved"] for r in evaluation))
